In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# elbow-gene-choose

Evaluate selected-feature counts and produce elbow plots and gene lists.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


In [ ]:
import pandas as pd
import os
import sys
import numpy as np
import h5py
import re
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
from tqdm import tqdm


MAPLE_BASE_OUTPUT_DIR = input_path("2-8.3-shanda/1-feature/1-5x")
H5_BASE_DIR = input_path("1-TMS-remove/2-restart")
HEADER_FILE = input_path("header.txt")
OUTPUT_CSV_PATH = output_path("2-8.3-shanda/1-feature/7-3feature-pcc_Fold4.csv")

TARGET_FOLD = "fold_4"
START_FEATURE_INDEX = 1011
AGE_MAPPING = {0: 1, 1: 3, 2: 18, 3: 21, 4: 24, 5: 30}


def map_labels(labels):
    return np.vectorize(AGE_MAPPING.get)(labels)

def get_gene_names():
    try:
        with open(HEADER_FILE, 'r') as f:
            return [l.strip().upper() for l in f if l.strip()]
    except Exception as e:
        print(f"Error loading header: {e}")
        return []

def parse_feature_file(path, all_genes):
    try:
        with open(path, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]
        if len(lines) != len(all_genes): return []
        return [all_genes[i] for i, val in enumerate(lines) if float(val) == 1.0]
    except: return []

def load_h5_data(tissue, all_genes):
    train_path = os.path.join(H5_BASE_DIR, tissue, 'train.h5')
    test_path = os.path.join(H5_BASE_DIR, tissue, 'test.h5')
    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        return None, None, None, None
    try:
        with h5py.File(train_path, 'r') as f:
            X_train = pd.DataFrame(f['data'][:], columns=all_genes)
            y_train = map_labels(f['label'][:, 0].astype(np.int8))
        with h5py.File(test_path, 'r') as f:
            X_test = pd.DataFrame(f['data'][:], columns=all_genes)
            y_test = map_labels(f['label'][:, 0].astype(np.int8))
        return X_train, y_train, X_test, y_test
    except Exception as e:
        return None, None, None, None

def run_regression_metrics(X_train, y_train, X_test, y_test):

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if len(np.unique(y_test)) < 2:
        pcc = np.nan
    else:
        pcc, _ = pearsonr(y_test, y_pred)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return pcc, mae, rmse


def main():
    all_genes = get_gene_names()
    if not all_genes: sys.exit("Failed to load gene header.")
    if not os.path.exists(H5_BASE_DIR): sys.exit(f"H5 Base Dir not found")

    tissues = sorted([d for d in os.listdir(H5_BASE_DIR) if os.path.isdir(os.path.join(H5_BASE_DIR, d))])
    all_results = []

    print(f"--- Starting Feature Evaluation (Raw Data -> CSV) ---")

    for tissue in tqdm(tissues, desc="Processing Tissues"):
        X_train_df, y_train, X_test_df, y_test = load_h5_data(tissue, all_genes)
        if X_train_df is None: continue

        base_cv_path = os.path.join(MAPLE_BASE_OUTPUT_DIR, tissue, "cv_folds")
        if not os.path.exists(base_cv_path): continue

        run_dir = os.path.join(base_cv_path, TARGET_FOLD, "run")
        if not os.path.exists(run_dir): continue

        feature_folders = []
        try:
            for d in os.listdir(run_dir):
                match = re.match(r'feature(\d+)', d)
                if match:
                    idx = int(match.group(1))
                    if idx <= START_FEATURE_INDEX:
                        feature_folders.append((idx, d))
        except: continue

        if not feature_folders: continue
        feature_folders.sort(key=lambda x: x[0], reverse=True)

        for idx, folder_name in feature_folders:
            feature_path = os.path.join(run_dir, folder_name, "feature.txt")
            if not os.path.exists(feature_path): continue

            selected_genes = parse_feature_file(feature_path, all_genes)
            gene_count = len(selected_genes)
            if gene_count == 0: continue

            valid_genes = [g for g in selected_genes if g in X_train_df.columns]
            if not valid_genes: continue

            pcc, mae, rmse = run_regression_metrics(
                X_train_df[valid_genes].values, y_train,
                X_test_df[valid_genes].values, y_test
            )

            all_results.append({
                'Tissue': tissue, 'Fold': TARGET_FOLD, 'Feature_Index': idx,
                'Gene_Count': gene_count, 'PCC': pcc, 'MAE': mae, 'RMSE': rmse
            })

    if all_results:
        df_out = pd.DataFrame(all_results)[['Tissue', 'Fold', 'Feature_Index', 'Gene_Count', 'PCC', 'MAE', 'RMSE']]
        os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
        df_out.to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"\n✅ 第一步完成！底层评估指标已成功保存至: {OUTPUT_CSV_PATH}")
    else:
        print("\nNo results generated.")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os
import sys
import numpy as np
import h5py
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from tqdm import tqdm


MAPLE_BASE_OUTPUT_DIR = input_path("2-8.3-shanda/1-feature/1-5x")
HEADER_FILE = input_path("header.txt")

OUTPUT_CSV_PATH = output_path("2-8.3-shanda/1-feature/7-3feature-pcc_Fold4.csv")


OUTPUT_PLOT_DIR = output_path("2-8.3-shanda/1-feature/9-3feature-pcc_Segmented_Knee")
OUTPUT_GENE_LIST_DIR = output_path("2-8.3-shanda/1-feature/9-3Final_Segmented_Genes")

TARGET_FOLD = "fold_4"


def get_gene_names():
    try:
        with open(HEADER_FILE, 'r') as f:
            return [l.strip().upper() for l in f if l.strip()]
    except: return []

def parse_feature_file(path, all_genes):
    try:
        with open(path, 'r') as f:
            lines = [l.strip() for l in f if l.strip()]
        return [all_genes[i] for i, val in enumerate(lines) if float(val) == 1.0]
    except Exception as e:
        print(f"Error parsing feature file: {e}")
        return []


def find_accurate_segmented_knee(df):
    df = df.sort_values(by='Gene_Count').drop_duplicates(subset=['Gene_Count'])
    x_raw = df['Gene_Count'].values.astype(float)
    y_raw = df['MAE'].values.astype(float)

    if len(x_raw) < 6: return None

    median_y = np.median(y_raw)
    mad = np.median(np.abs(y_raw - median_y))
    valid_mask = np.abs(y_raw - median_y) < 5 * mad
    x_cl, y_cl = x_raw[valid_mask], y_raw[valid_mask]

    min_mae_idx = np.argmin(y_cl)
    search_end = max(4, min_mae_idx)
    x_sub, y_sub = x_cl[:search_end + 1], y_cl[:search_end + 1]

    best_rss = np.inf
    best_idx = 0
    for i in range(2, len(x_sub) - 2):
        rss = (np.sum((y_sub[:i+1] - LinearRegression().fit(x_sub[:i+1].reshape(-1,1), y_sub[:i+1]).predict(x_sub[:i+1].reshape(-1,1)))**2) +
               np.sum((y_sub[i:] - LinearRegression().fit(x_sub[i:].reshape(-1,1), y_sub[i:]).predict(x_sub[i:].reshape(-1,1)))**2))
        if rss < best_rss:
            best_rss, best_idx = rss, i

    knee_genes_count = x_sub[best_idx]
    row = df[df['Gene_Count'] == knee_genes_count].iloc[0]
    return {
        'Genes': int(knee_genes_count),
        'MAE': row['MAE'],
        'PCC': row['PCC'],
        'RMSE': row.get('RMSE', np.nan),
        'Feature_Index': int(row['Feature_Index'])
    }


def plot_comprehensive_metrics(df, knee_dec, output_dir, tissue):
    df = df.sort_values('Gene_Count')
    metrics = [m for m in ['PCC', 'MAE', 'RMSE'] if m in df.columns]
    colors = {'PCC': '#4DBBD5', 'MAE': '#E64B35', 'RMSE': '#00A087'}

    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    if len(metrics) == 1: axes = [axes]

    for i, metric in enumerate(metrics):
        ax = axes[i]
        sns.lineplot(data=df, x='Gene_Count', y=metric, ax=ax, color=colors[metric], alpha=0.5, lw=2)


        knee_val = knee_dec[metric]
        ax.scatter(knee_dec['Genes'], knee_val, color='black', s=130, marker='*', zorder=10, label='Knee')
        ax.annotate(f"Knee: {knee_val:.3f}\n({knee_dec['Genes']} G)",
                    xy=(knee_dec['Genes'], knee_val), xytext=(25, 20), textcoords='offset points',
                    arrowprops=dict(arrowstyle="->", color='black'), fontsize=9, fontweight='bold')


        if metric == 'PCC':
            best_val = df[metric].max()
            best_row = df.loc[df[metric].idxmax()]
            best_label = "Max"
        else:
            best_val = df[metric].min()
            best_row = df.loc[df[metric].idxmin()]
            best_label = "Min"

        ax.scatter(best_row['Gene_Count'], best_val, color='red', s=70, marker='o', edgecolors='white', zorder=9, label=f'Global {best_label}')
        ax.annotate(f"{best_label}: {best_val:.3f}\n({int(best_row['Gene_Count'])} G)",
                    xy=(best_row['Gene_Count'], best_val), xytext=(25, -35), textcoords='offset points',
                    arrowprops=dict(arrowstyle="->", color='red'), fontsize=9, color='red', fontweight='bold')

        ax.set_title(f"{metric} Trends", fontweight='bold', fontsize=12)
        ax.set_xlabel("Number of Genes")
        ax.grid(True, linestyle='--', alpha=0.3)
        if i == 0: ax.legend(loc='lower right')

    plt.suptitle(f"Selection Analysis: {tissue}", fontsize=15, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{tissue}_Metrics_Annotated.pdf"), dpi=300, bbox_inches='tight')
    plt.close()


def main():
    print("--- Starting Accurate Feature Selection & Annotation ---")
    all_genes = get_gene_names()
    if not all_genes: sys.exit("Error: Could not load header.txt")
    if not os.path.exists(OUTPUT_CSV_PATH): sys.exit("Error: Metrics CSV not found")

    df_all = pd.read_csv(OUTPUT_CSV_PATH)
    os.makedirs(OUTPUT_PLOT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_GENE_LIST_DIR, exist_ok=True)

    summary_list = []

    for tissue in tqdm(df_all['Tissue'].unique(), desc="Processing Tissues"):
        t_df = df_all[df_all['Tissue'] == tissue].copy()

        decision = find_accurate_segmented_knee(t_df)

        if decision:
            base_run_path = os.path.join(MAPLE_BASE_OUTPUT_DIR, tissue, "cv_folds", TARGET_FOLD, "run")
            target_idx = decision['Feature_Index']

            feat_file_path = None
            if os.path.exists(base_run_path):
                for folder in os.listdir(base_run_path):
                    match = re.match(r'feature(\d+)', folder)
                    if match and int(match.group(1)) == target_idx:
                        feat_file_path = os.path.join(base_run_path, folder, "feature.txt")
                        break

            if feat_file_path and os.path.exists(feat_file_path):
                gene_list = parse_feature_file(feat_file_path, all_genes)

                if gene_list:
                    save_name = f"{tissue}_Knee_Genes_{len(gene_list)}.txt"
                    with open(os.path.join(OUTPUT_GENE_LIST_DIR, save_name), 'w') as f:
                        for g in gene_list: f.write(f"{g}\n")

                    plot_comprehensive_metrics(t_df, decision, OUTPUT_PLOT_DIR, tissue)

                    summary_list.append({
                        'Tissue': tissue,
                        'Selected_Genes': len(gene_list),
                        'MAE': decision['MAE'],
                        'PCC': decision['PCC']
                    })

    if summary_list:
        pd.DataFrame(summary_list).to_csv(os.path.join(OUTPUT_GENE_LIST_DIR, "Final_Knee_Summary.csv"), index=False)
        print(f"\n✅ 第二步完成！决策图表与终极基因列表(.txt)已存入: {OUTPUT_GENE_LIST_DIR}")

if __name__ == "__main__":
    main()